# Extracción de Grafos de Conocimiento con Claude

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/graphrag/03-extraccion-grafos-con-llm.ipynb)

Extracción automática de entidades y relaciones desde texto con Claude y almacenamiento en Neo4j.

## ¿Qué aprenderás?

- Usar `tool_use` de Claude para extracción estructurada de información
- Modelar grafos con Pydantic (tipos estrictos, validación)
- Deduplicar entidades con embeddings semánticos
- Almacenar grafos en Neo4j y consultar con Cypher
- Escalar el pipeline a múltiples documentos

## Arquitectura

```
Texto libre
    │
    ▼
Claude (tool_use: extraer_grafo)
    │
    ▼
GrafoConocimiento (Pydantic)
    │
    ├── Deduplicación con embeddings
    │
    └── Neo4j / NetworkX (almacenamiento)
```

In [ ]:
# Instalar dependencias
# pip install anthropic pydantic neo4j numpy scikit-learn networkx matplotlib

import os
import json
import re
from typing import Optional

import anthropic
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pydantic import BaseModel, Field, field_validator

# Cliente de Anthropic
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
MODO_SIMULADO = not ANTHROPIC_API_KEY

if not MODO_SIMULADO:
    cliente_anthropic = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    print("Cliente Anthropic conectado.")
else:
    cliente_anthropic = None
    print("ANTHROPIC_API_KEY no encontrada. Modo simulado activado.")

print(f"Pydantic v{BaseModel.__module__.split('.')[0]} cargado")

## 1. Modelos Pydantic: Entidad, Relacion, GrafoConocimiento

Definimos los tipos de datos con validación estricta. Estos modelos se pasan a Claude como JSON Schema en el `tool_use`.

In [ ]:
TIPOS_ENTIDAD = [
    "PERSONA", "ORGANIZACION", "TECNOLOGIA", "CONCEPTO",
    "LUGAR", "EVENTO", "PRODUCTO", "OTRO",
]

TIPOS_RELACION = [
    "TRABAJA_EN", "FUNDO", "DESARROLLA", "COLABORA_CON",
    "PERTENECE_A", "CREÓ", "COMPITE_CON", "FINANCIA",
    "UTILIZA", "ES_PARTE_DE", "RELACIONADO_CON", "OTRO",
]


class Entidad(BaseModel):
    """Una entidad nombrada extraída del texto."""

    nombre: str = Field(..., description="Nombre canónico de la entidad", min_length=1)
    tipo: str = Field(..., description="Tipo de entidad")
    descripcion: str = Field(default="", description="Descripción breve extraída del texto")
    menciones: list[str] = Field(default_factory=list, description="Formas alternativas del nombre")

    @field_validator("tipo")
    @classmethod
    def validar_tipo(cls, v: str) -> str:
        v_upper = v.upper()
        return v_upper if v_upper in TIPOS_ENTIDAD else "OTRO"

    @field_validator("nombre")
    @classmethod
    def limpiar_nombre(cls, v: str) -> str:
        return v.strip().title()


class Relacion(BaseModel):
    """Una relación entre dos entidades."""

    origen: str = Field(..., description="Nombre de la entidad origen")
    tipo: str = Field(..., description="Tipo de relación")
    destino: str = Field(..., description="Nombre de la entidad destino")
    descripcion: str = Field(default="", description="Descripción de la relación")
    fuerza: float = Field(default=1.0, ge=0.0, le=1.0, description="Confianza 0-1")

    @field_validator("tipo")
    @classmethod
    def validar_tipo(cls, v: str) -> str:
        v_upper = v.upper().replace(" ", "_")
        return v_upper if v_upper in TIPOS_RELACION else "RELACIONADO_CON"


class GrafoConocimiento(BaseModel):
    """Grafo de conocimiento extraído de un texto."""

    entidades: list[Entidad] = Field(default_factory=list)
    relaciones: list[Relacion] = Field(default_factory=list)
    resumen: str = Field(default="", description="Resumen del contenido del texto")

    def a_networkx(self) -> nx.DiGraph:
        """Convierte el grafo Pydantic a un grafo NetworkX."""
        G = nx.DiGraph()
        for ent in self.entidades:
            G.add_node(ent.nombre, tipo=ent.tipo, descripcion=ent.descripcion)
        for rel in self.relaciones:
            if rel.origen in G.nodes and rel.destino in G.nodes:
                G.add_edge(rel.origen, rel.destino, tipo=rel.tipo, fuerza=rel.fuerza)
        return G

    def estadisticas(self) -> dict:
        """Retorna estadísticas del grafo."""
        tipos_ent = {}
        for e in self.entidades:
            tipos_ent[e.tipo] = tipos_ent.get(e.tipo, 0) + 1
        return {
            "entidades_total": len(self.entidades),
            "relaciones_total": len(self.relaciones),
            "tipos_entidad": tipos_ent,
        }


print("Modelos Pydantic definidos:")
print(f"  Entidad: {list(Entidad.model_fields.keys())}")
print(f"  Relacion: {list(Relacion.model_fields.keys())}")
print(f"  GrafoConocimiento: {list(GrafoConocimiento.model_fields.keys())}")

## 2. Función `extraer_grafo()` con `tool_use` de Claude

Usamos el mecanismo de `tool_use` de Claude para forzar una salida estructurada en JSON
que coincide exactamente con nuestros modelos Pydantic.

In [ ]:
# Definición del tool para Claude
HERRAMIENTA_EXTRACCION = {
    "name": "guardar_grafo_conocimiento",
    "description": (
        "Guarda el grafo de conocimiento extraído del texto. "
        "Extrae TODAS las entidades nombradas (personas, organizaciones, tecnologías, conceptos) "
        "y las relaciones explícitas entre ellas."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "entidades": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "nombre": {"type": "string"},
                        "tipo": {"type": "string", "enum": TIPOS_ENTIDAD},
                        "descripcion": {"type": "string"},
                        "menciones": {"type": "array", "items": {"type": "string"}},
                    },
                    "required": ["nombre", "tipo"],
                },
            },
            "relaciones": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "origen": {"type": "string"},
                        "tipo": {"type": "string", "enum": TIPOS_RELACION},
                        "destino": {"type": "string"},
                        "descripcion": {"type": "string"},
                        "fuerza": {"type": "number", "minimum": 0, "maximum": 1},
                    },
                    "required": ["origen", "tipo", "destino"],
                },
            },
            "resumen": {"type": "string"},
        },
        "required": ["entidades", "relaciones", "resumen"],
    },
}


def extraer_grafo(texto: str, modo_simulado: bool = MODO_SIMULADO) -> GrafoConocimiento:
    """
    Extrae un grafo de conocimiento de un texto usando Claude tool_use.
    
    Args:
        texto: texto del que extraer entidades y relaciones
        modo_simulado: si True, devuelve un grafo de ejemplo sin llamar a la API
    
    Returns:
        GrafoConocimiento validado con Pydantic
    """
    if modo_simulado:
        return _grafo_simulado(texto)

    respuesta = cliente_anthropic.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=4096,
        tools=[HERRAMIENTA_EXTRACCION],
        tool_choice={"type": "tool", "name": "guardar_grafo_conocimiento"},
        messages=[
            {
                "role": "user",
                "content": (
                    "Analiza el siguiente texto y extrae TODAS las entidades nombradas "
                    "(personas, organizaciones, tecnologías, conceptos, lugares) y las "
                    "relaciones entre ellas. Sé exhaustivo.\n\n"
                    f"TEXTO:\n{texto}"
                ),
            }
        ],
    )

    # Extraer el resultado del tool_use
    tool_result = next(
        (b for b in respuesta.content if b.type == "tool_use"), None
    )
    if not tool_result:
        raise ValueError("Claude no devolvió un resultado de tool_use")

    return GrafoConocimiento(**tool_result.input)


def _grafo_simulado(texto: str) -> GrafoConocimiento:
    """Genera un grafo simulado para pruebas sin API."""
    return GrafoConocimiento(
        entidades=[
            Entidad(nombre="Anthropic", tipo="ORGANIZACION",
                    descripcion="Empresa de IA fundada en 2021"),
            Entidad(nombre="Dario Amodei", tipo="PERSONA",
                    descripcion="CEO y cofundador de Anthropic"),
            Entidad(nombre="Daniela Amodei", tipo="PERSONA",
                    descripcion="Presidenta y cofundadora de Anthropic"),
            Entidad(nombre="Claude", tipo="PRODUCTO",
                    descripcion="Asistente IA desarrollado por Anthropic"),
            Entidad(nombre="Openai", tipo="ORGANIZACION",
                    descripcion="Empresa de IA competidora"),
            Entidad(nombre="Constitutional Ai", tipo="TECNOLOGIA",
                    descripcion="Técnica de entrenamiento de Anthropic"),
        ],
        relaciones=[
            Relacion(origen="Dario Amodei", tipo="FUNDO", destino="Anthropic",
                     descripcion="Cofundador y CEO", fuerza=1.0),
            Relacion(origen="Daniela Amodei", tipo="FUNDO", destino="Anthropic",
                     descripcion="Cofundadora y Presidenta", fuerza=1.0),
            Relacion(origen="Anthropic", tipo="DESARROLLA", destino="Claude",
                     descripcion="Desarrolla y mantiene Claude", fuerza=1.0),
            Relacion(origen="Anthropic", tipo="UTILIZA", destino="Constitutional Ai",
                     descripcion="Metodología de entrenamiento", fuerza=0.9),
            Relacion(origen="Dario Amodei", tipo="TRABAJA_EN", destino="Openai",
                     descripcion="Antes de fundar Anthropic", fuerza=0.8),
        ],
        resumen="Anthropic es una empresa de IA fundada por Dario y Daniela Amodei, "
                "ex empleados de OpenAI, que desarrolla el asistente Claude.",
    )


print("Función extraer_grafo() definida")
print(f"Modo: {'simulado' if MODO_SIMULADO else 'API real (claude-haiku-4-5-20251001)'}")

## 3. Texto de prueba y extracción en vivo

In [ ]:
TEXTO_PRUEBA = """
Anthropic fue fundada en 2021 por Dario Amodei y Daniela Amodei, junto con otros ex empleados
de OpenAI. Dario, que fue VP de investigación en OpenAI, y Daniela, directora de operaciones,
decidieron crear una empresa enfocada en la seguridad de la IA.

La empresa desarrolla Claude, un asistente de inteligencia artificial que compite directamente
con ChatGPT de OpenAI y Gemini de Google. Claude usa una técnica propia llamada Constitutional AI
para aprender a comportarse de forma segura y útil, desarrollada por el equipo de investigación
de Anthropic liderado por Chris Olah.

En 2023, Amazon invirtió 4.000 millones de dólares en Anthropic, convirtiéndose en su principal
socio cloud. Además, Anthropic tiene acuerdos de colaboración con Google Cloud y Salesforce.
"""

print("Extrayendo grafo del texto...")
grafo = extraer_grafo(TEXTO_PRUEBA)

print("\n=== GRAFO EXTRAÍDO ===")
stats = grafo.estadisticas()
print(f"Entidades: {stats['entidades_total']}")
print(f"Relaciones: {stats['relaciones_total']}")
print(f"Tipos: {stats['tipos_entidad']}")

print("\n--- Entidades ---")
for ent in grafo.entidades:
    print(f"  [{ent.tipo}] {ent.nombre}: {ent.descripcion}")

print("\n--- Relaciones ---")
for rel in grafo.relaciones:
    print(f"  {rel.origen} --[{rel.tipo}]--> {rel.destino} (fuerza: {rel.fuerza})")

print(f"\n--- Resumen ---")
print(f"  {grafo.resumen}")

## 4. Deduplicación de entidades con embeddings

In [ ]:
def similitud_coseno(v1: list[float], v2: list[float]) -> float:
    """Calcula la similitud coseno entre dos vectores."""
    a, b = np.array(v1), np.array(v2)
    norma = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / norma) if norma > 0 else 0.0


def embedding_simple(texto: str) -> list[float]:
    """
    Embedding simple basado en frecuencia de caracteres (para simulación).
    En producción usa sentence-transformers o la API de embeddings.
    """
    texto = texto.lower()
    vec = [texto.count(c) / max(len(texto), 1) for c in "abcdefghijklmnopqrstuvwxyz"]
    return vec


def deduplicar_entidades(
    entidades: list[Entidad],
    umbral_similitud: float = 0.98,
) -> list[Entidad]:
    """
    Fusiona entidades duplicadas o casi duplicadas.
    
    Estrategia:
    1. Normalizar nombres (minúsculas, sin puntuación extra)
    2. Calcular embeddings de los nombres
    3. Fusionar entidades con similitud > umbral
    """
    if not entidades:
        return []

    # Normalizar nombres
    def normalizar(nombre: str) -> str:
        return re.sub(r"[^a-z0-9 ]", "", nombre.lower()).strip()

    embeddings = [embedding_simple(normalizar(e.nombre)) for e in entidades]

    grupos_fusionados: list[list[int]] = []
    visitados = set()

    for i, ent_i in enumerate(entidades):
        if i in visitados:
            continue
        grupo = [i]
        for j in range(i + 1, len(entidades)):
            if j in visitados:
                continue
            # Comprobación exacta primero
            if normalizar(ent_i.nombre) == normalizar(entidades[j].nombre):
                grupo.append(j)
                visitados.add(j)
                continue
            # Similitud coseno para variantes
            sim = similitud_coseno(embeddings[i], embeddings[j])
            if sim >= umbral_similitud and ent_i.tipo == entidades[j].tipo:
                grupo.append(j)
                visitados.add(j)
        visitados.add(i)
        grupos_fusionados.append(grupo)

    # Fusionar grupos en una sola entidad
    resultado = []
    for grupo in grupos_fusionados:
        entidad_principal = entidades[grupo[0]]
        menciones_extra = []
        for idx in grupo[1:]:
            menciones_extra.append(entidades[idx].nombre)
            menciones_extra.extend(entidades[idx].menciones)
        entidad_fusionada = Entidad(
            nombre=entidad_principal.nombre,
            tipo=entidad_principal.tipo,
            descripcion=entidad_principal.descripcion,
            menciones=list(set(entidad_principal.menciones + menciones_extra)),
        )
        resultado.append(entidad_fusionada)

    return resultado


# Demostración de deduplicación
entidades_con_duplicados = grafo.entidades + [
    Entidad(nombre="ANTHROPIC", tipo="ORGANIZACION", descripcion="Empresa duplicada"),
    Entidad(nombre="claude", tipo="PRODUCTO", descripcion="Producto duplicado"),
]

print(f"Antes de deduplicar: {len(entidades_con_duplicados)} entidades")
deduplicadas = deduplicar_entidades(entidades_con_duplicados)
print(f"Después de deduplicar: {len(deduplicadas)} entidades")

print("\nEntidades finales:")
for e in deduplicadas:
    menciones_str = f" (también: {', '.join(e.menciones)})" if e.menciones else ""
    print(f"  [{e.tipo}] {e.nombre}{menciones_str}")

## 5. Guardar grafo en Neo4j (o NetworkX) y consultar

In [ ]:
class AlmacenGrafo:
    """Almacena y consulta grafos de conocimiento (Neo4j o modo simulado NetworkX)."""

    def __init__(self, modo_simulado: bool = True):
        self.modo_simulado = modo_simulado
        self._G = nx.DiGraph()  # Grafo acumulado

    def guardar(self, grafo: GrafoConocimiento, fuente: str = "") -> int:
        """Guarda el grafo de conocimiento. Retorna número de nodos añadidos."""
        nuevos = 0
        for ent in grafo.entidades:
            if ent.nombre not in self._G:
                nuevos += 1
            self._G.add_node(
                ent.nombre,
                tipo=ent.tipo,
                descripcion=ent.descripcion,
                fuente=fuente,
            )
        for rel in grafo.relaciones:
            self._G.add_edge(
                rel.origen, rel.destino,
                tipo=rel.tipo,
                fuerza=rel.fuerza,
                descripcion=rel.descripcion,
            )
        return nuevos

    def buscar_entidad(self, nombre: str) -> dict:
        """Retorna la entidad y sus vecinos directos."""
        if nombre not in self._G:
            return {"error": f"Entidad '{nombre}' no encontrada"}
        datos = dict(self._G.nodes[nombre])
        salientes = [
            {"destino": d, "tipo": self._G.edges[nombre, d].get("tipo", "")}
            for d in self._G.successors(nombre)
        ]
        entrantes = [
            {"origen": o, "tipo": self._G.edges[o, nombre].get("tipo", "")}
            for o in self._G.predecessors(nombre)
        ]
        return {"entidad": nombre, "datos": datos, "relaciones_salientes": salientes, "relaciones_entrantes": entrantes}

    def estadisticas(self) -> dict:
        return {
            "nodos_totales": self._G.number_of_nodes(),
            "relaciones_totales": self._G.number_of_edges(),
        }

    def visualizar(self, titulo: str = "Grafo de Conocimiento"):
        """Visualiza el grafo con matplotlib."""
        color_map = {
            "PERSONA": "#4A90D9", "ORGANIZACION": "#2ECC71",
            "TECNOLOGIA": "#E67E22", "PRODUCTO": "#9B59B6",
            "CONCEPTO": "#E74C3C", "OTRO": "#95A5A6",
        }
        G = self._G
        node_colors = [
            color_map.get(G.nodes[n].get("tipo", "OTRO"), "#95A5A6")
            for n in G.nodes
        ]
        plt.figure(figsize=(14, 9))
        pos = nx.spring_layout(G, seed=42, k=3)
        nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1000, alpha=0.9)
        nx.draw_networkx_labels(G, pos, font_size=8, font_color="white", font_weight="bold")
        edge_labels = {(u, v): d.get("tipo", "") for u, v, d in G.edges(data=True)}
        nx.draw_networkx_edges(G, pos, edge_color="#CCCCCC", arrows=True, arrowsize=15,
                               width=1.5, connectionstyle="arc3,rad=0.1")
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7)
        leyenda = [mpatches.Patch(color=v, label=k) for k, v in color_map.items() if k != "OTRO"]
        plt.legend(handles=leyenda, loc="upper left", fontsize=9)
        plt.title(titulo, fontsize=14, fontweight="bold")
        plt.axis("off")
        plt.tight_layout()
        plt.show()


# Guardar y consultar
almacen = AlmacenGrafo(modo_simulado=True)
nuevos = almacen.guardar(grafo, fuente="texto_prueba")

print(f"Grafo guardado: {nuevos} entidades nuevas")
print(f"Estadísticas: {almacen.estadisticas()}")

print("\n=== Consulta: información sobre Anthropic ===")
resultado = almacen.buscar_entidad("Anthropic")
print(f"Datos: {resultado['datos']}")
print(f"Relaciones salientes: {resultado['relaciones_salientes']}")
print(f"Relaciones entrantes: {resultado['relaciones_entrantes']}")

almacen.visualizar("Grafo: Anthropic y su ecosistema")

## 6. Pipeline completo sobre múltiples textos

In [ ]:
CORPUS = [
    {
        "id": "doc_001",
        "titulo": "Anthropic y Claude",
        "texto": TEXTO_PRUEBA,
    },
    {
        "id": "doc_002",
        "titulo": "OpenAI y GPT",
        "texto": """
        OpenAI fue fundada en 2015 por Sam Altman, Elon Musk y Greg Brockman.
        Desarrolla los modelos GPT-4 y GPT-4o, así como el asistente ChatGPT.
        Microsoft invirtió 13.000 millones de dólares en la empresa.
        OpenAI compite directamente con Anthropic en el mercado de asistentes de IA.
        """,
    },
    {
        "id": "doc_003",
        "titulo": "Google y Gemini",
        "texto": """
        Google DeepMind, liderada por Demis Hassabis, desarrolla Gemini.
        Gemini compite con Claude de Anthropic y GPT-4 de OpenAI.
        Google Cloud tiene acuerdos con Anthropic para despliegue de Claude.
        """,
    },
]


def procesar_corpus(
    documentos: list[dict],
    modo_simulado: bool = True,
) -> AlmacenGrafo:
    """Procesa un corpus completo y construye un grafo acumulado."""
    almacen_global = AlmacenGrafo(modo_simulado=modo_simulado)
    todas_entidades = []

    for i, doc in enumerate(documentos):
        print(f"[{i+1}/{len(documentos)}] Procesando: {doc['titulo']}...")
        grafo_doc = extraer_grafo(doc["texto"], modo_simulado=modo_simulado)

        # Deduplicar antes de guardar
        todas_entidades.extend(grafo_doc.entidades)
        entidades_dedup = deduplicar_entidades(grafo_doc.entidades)
        grafo_doc_dedup = GrafoConocimiento(
            entidades=entidades_dedup,
            relaciones=grafo_doc.relaciones,
            resumen=grafo_doc.resumen,
        )

        nuevos = almacen_global.guardar(grafo_doc_dedup, fuente=doc["id"])
        stats = grafo_doc.estadisticas()
        print(f"   Entidades: {stats['entidades_total']}, "
              f"Relaciones: {stats['relaciones_total']}, "
              f"Nuevas: {nuevos}")

    print(f"\nGrafo global: {almacen_global.estadisticas()}")
    return almacen_global


almacen_global = procesar_corpus(CORPUS, modo_simulado=MODO_SIMULADO)
almacen_global.visualizar("Grafo de conocimiento global del corpus")

## Resumen

### Pipeline de extracción de grafos

| Paso | Herramienta | Clave |
|------|-------------|-------|
| Definición de esquema | Pydantic + JSON Schema | Tipos validados y documentados |
| Extracción | Claude `tool_use` | Salida garantizada en JSON |
| Deduplicación | Embeddings + similitud coseno | Evita duplicados entre documentos |
| Almacenamiento | Neo4j / NetworkX | Persistencia y consulta |

### Buenas prácticas

- **Usa `tool_use`**, no instrucciones de formato en el prompt: más robusto y sin parsing frágil
- **Deduplica siempre** antes de insertar en Neo4j para evitar nodos fantasma
- **Registra la fuente** de cada entidad para trazabilidad
- **Itera el grafo**: ejecuta múltiples pasadas para capturar relaciones implícitas

### Próximo paso

El **Notebook 04** compara GraphRAG vs VectorRAG en producción y muestra cómo construir un sistema RAG híbrido.